# 06 — Reliability: Data Quality Expectations

Persists pass/fail for ~10 data-quality rules into a `dq_results` Delta table
you can graph, alert on, and SLA against.

## Why this matters — Well-Architected Framework

| WAF pillar | What this notebook does, and why it's good |
|------------|---------------------------------------------|
| **Reliability** | The `CHECK` constraints in 03 block obvious junk. This notebook handles the *multi-table* rules they can't express: referential integrity, freshness, row-count anomalies. Two layers of defence. |
| **Operational Excellence** | Results are in Delta → graphable in Lakeview, alertable via Lakeflow, auditable forever. A DQ dashboard tile is ~15 minutes of work from here. |
| **Data Governance** | Every rule has a `severity` (`warn` vs `error`) and `notes` — auditors love this. |

> This pattern generalizes — in production, swap the hand-rolled `run_check`
> helper for **Lakeflow Declarative Pipelines (DLT) `EXPECT` clauses** or
> the open-source **Databricks Labs DQX** library. The shape of the
> `dq_results` table stays the same; only how you *populate* it changes.


In [1]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os, uuid
from datetime import datetime

load_dotenv()

spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG    = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA     = os.getenv("UC_SCHEMA",  "mlb_gumbo_waf")
BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
SILVER_SCHEMA = f"{UC_SCHEMA}_silver"
GOLD_SCHEMA   = f"{UC_SCHEMA}_gold"

B = f"{UC_CATALOG}.{BRONZE_SCHEMA}"
S = f"{UC_CATALOG}.{SILVER_SCHEMA}"
G = f"{UC_CATALOG}.{GOLD_SCHEMA}"
print(f"Targets: {B}, {S}, {G}")


Targets: alexander_booth.mlb_gumbo_waf_bronze, alexander_booth.mlb_gumbo_waf_silver, alexander_booth.mlb_gumbo_waf_gold


In [2]:
dq_table = f"{G}.dq_results"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {dq_table} (
        run_id         STRING,
        run_time       TIMESTAMP,
        layer          STRING,
        table_name     STRING,
        rule_name      STRING,
        severity       STRING,
        failing_rows   BIGINT,
        total_rows     BIGINT,
        passed         BOOLEAN,
        notes          STRING
    )
    CLUSTER BY (run_time)
    COMMENT 'Per-run data quality expectation results for the mlb_gumbo_waf pipeline.'
""")
print(f"DQ results table: {dq_table}")


DQ results table: alexander_booth.mlb_gumbo_waf_gold.dq_results


In [3]:
RUN_ID   = str(uuid.uuid4())
RUN_TIME = datetime.utcnow().isoformat()

def run_check(layer, table_name, rule_name, predicate_fail, severity="warn", notes=""):
    """predicate_fail = SQL predicate that matches FAILING rows."""
    failing = spark.sql(f"SELECT COUNT(*) c FROM {table_name} WHERE {predicate_fail}").collect()[0]["c"]
    total   = spark.sql(f"SELECT COUNT(*) c FROM {table_name}").collect()[0]["c"]
    passed  = failing == 0
    spark.sql(f"""
        INSERT INTO {dq_table} VALUES (
            '{RUN_ID}', TIMESTAMP'{RUN_TIME}', '{layer}', '{table_name}',
            '{rule_name}', '{severity}', {failing}, {total}, {str(passed).lower()},
            '{notes.replace("'", "''")}'
        )
    """)
    icon = "PASS" if passed else ("FAIL" if severity == "error" else "WARN")
    print(f"  [{icon}] {rule_name}: {failing}/{total} failing")
    return passed

print(f"Run {RUN_ID}")


Run 9a4ec177-e657-400d-9a10-c47747fa7dfb


/var/folders/xn/_5vq53hj5ld7v66jgsmc64f40000gp/T/ipykernel_72117/3829098428.py:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  RUN_TIME = datetime.utcnow().isoformat()


## Bronze-layer checks

In [4]:
run_check("bronze", f"{B}.raw_gumbo", "bronze_no_null_payload",
          "data IS NULL", severity="error",
          notes="Raw VARIANT payload must be non-null")

run_check("bronze", f"{B}.raw_gumbo", "bronze_no_unparsable_files",
          "source_file_size = 0", severity="warn",
          notes="Zero-byte files indicate a broken upstream fetch")

# Freshness: at least one row landed in the last 7 days
run_check("bronze", f"{B}.raw_gumbo", "bronze_freshness_7d",
          "file_batch_time > date_sub(current_timestamp(), 7)",
          severity="warn",
          notes="Inverted: we flag rows older than 7d. Only useful if pipeline runs on a schedule.")


  [PASS] bronze_no_null_payload: 0/2477 failing


  [PASS] bronze_no_unparsable_files: 0/2477 failing


  [WARN] bronze_freshness_7d: 2477/2477 failing


False

## Silver-layer checks

In [5]:
# Referential integrity: every pitch's game_sk must exist in game_data
run_check("silver", f"{S}.pitch_data p", "silver_pitch_game_ref",
          f"NOT EXISTS (SELECT 1 FROM {S}.game_data g WHERE g.game_sk = p.game_sk)",
          severity="error",
          notes="FK integrity between pitch_data and game_data")

# Uniqueness: game_sk PK
run_check("silver", f"{S}.game_data g", "silver_game_sk_unique",
          f"game_sk IN (SELECT game_sk FROM {S}.game_data GROUP BY game_sk HAVING COUNT(*) > 1)",
          severity="error", notes="PK uniqueness")

# Domain: pitch_type_code should be known set (A/F/C/...) — missing or gibberish is OK
run_check("silver", f"{S}.pitch_data", "silver_pitch_velo_range",
          "start_speed_mph < 30 OR start_speed_mph > 110",
          severity="warn",
          notes="Velocity in plausible human range; NULLs excluded by the predicate")


  [PASS] silver_pitch_game_ref: 0/724180 failing


  [PASS] silver_game_sk_unique: 0/2477 failing


  [WARN] silver_pitch_velo_range: 1/724180 failing


False

## Gold-layer checks

In [6]:
# Every fact_games row should resolve to a valid date dim
run_check("gold", f"{G}.fact_games fg", "gold_fact_games_date_fk",
          f"NOT EXISTS (SELECT 1 FROM {G}.dim_date d WHERE d.date_sk = fg.date_sk)",
          severity="error", notes="FK to dim_date")

# fact_pitches -> fact_games integrity
run_check("gold", f"{G}.fact_pitches p", "gold_fact_pitches_game_fk",
          f"NOT EXISTS (SELECT 1 FROM {G}.fact_games g WHERE g.game_sk = p.game_sk)",
          severity="error", notes="FK to fact_games")

# Pitch count reasonableness (a game typically 200-400 pitches; flag outliers)
run_check("gold",
          f"(SELECT game_sk, COUNT(*) AS n FROM {G}.fact_pitches GROUP BY game_sk) g",
          "gold_pitch_count_reasonable",
          "n < 50 OR n > 800",
          severity="warn",
          notes="Games should have ~200-400 pitches; flags suspicious outliers")


  [PASS] gold_fact_games_date_fk: 0/2477 failing


  [PASS] gold_fact_pitches_game_fk: 0/724180 failing


  [PASS] gold_pitch_count_reasonable: 0/2477 failing


True

## Summarize

In [7]:
print("\nLatest run summary:")
spark.sql(f"""
    SELECT layer, rule_name, severity,
           failing_rows, total_rows, passed
    FROM {dq_table}
    WHERE run_id = '{RUN_ID}'
    ORDER BY layer, rule_name
""").show(30, truncate=False)

print("\nPass rate over recent runs:")
spark.sql(f"""
    SELECT run_id, run_time,
           COUNT(*)                                AS checks_run,
           SUM(CASE WHEN passed THEN 1 ELSE 0 END) AS checks_passed,
           SUM(CASE WHEN NOT passed AND severity='error' THEN 1 ELSE 0 END) AS hard_failures
    FROM {dq_table}
    GROUP BY run_id, run_time
    ORDER BY run_time DESC
    LIMIT 10
""").show(truncate=False)



Latest run summary:


+------+---------------------------+--------+------------+----------+------+
|layer |rule_name                  |severity|failing_rows|total_rows|passed|
+------+---------------------------+--------+------------+----------+------+
|bronze|bronze_freshness_7d        |warn    |2477        |2477      |false |
|bronze|bronze_no_null_payload     |error   |0           |2477      |true  |
|bronze|bronze_no_unparsable_files |warn    |0           |2477      |true  |
|gold  |gold_fact_games_date_fk    |error   |0           |2477      |true  |
|gold  |gold_fact_pitches_game_fk  |error   |0           |724180    |true  |
|gold  |gold_pitch_count_reasonable|warn    |0           |2477      |true  |
|silver|silver_game_sk_unique      |error   |0           |2477      |true  |
|silver|silver_pitch_game_ref      |error   |0           |724180    |true  |
|silver|silver_pitch_velo_range    |warn    |1           |724180    |false |
+------+---------------------------+--------+------------+----------+------+

+------------------------------------+--------------------------+----------+-------------+-------------+
|run_id                              |run_time                  |checks_run|checks_passed|hard_failures|
+------------------------------------+--------------------------+----------+-------------+-------------+
|9a4ec177-e657-400d-9a10-c47747fa7dfb|2026-04-21 20:56:20.033543|9         |7            |0            |
+------------------------------------+--------------------------+----------+-------------+-------------+

